# 🩺 UBUZIMA AI — Kinyarwanda Voice Health Assistant

**Production-quality modular architecture** running entirely in Google Colab.

| Component | Model | Role |
|---|---|---|
| **Ears** (ASR) | `akera/whisper-large-v3-kin-200h-v2` | Speech → Kinyarwanda text |
| **Brain** (LLM) | Anthropic Claude (`claude-sonnet-4-6`) | Question → Kinyarwanda answer |
| **Mouth** (TTS) | `C4IR-RW/kinya-flex-tts` → `facebook/mms-tts-kin` fallback | Text → spoken Kinyarwanda (3 voices) |

## Notebook structure

| Cell | Module | Purpose |
|---|---|---|
| 1 | Install | All dependencies in one shot |
| 2 | Config | Paths, model IDs, constants |
| 3 | Auth | API keys (Anthropic, HuggingFace) |
| 4 | ASR module | `SpeechRecognizer` class |
| 5 | LLM module | `ClaudeAssistant` class |
| 6 | TTS module | `KinyaFlexTTS` with MMS-TTS fallback |
| 7 | Pipeline | `UbuzimaPipeline` orchestrator |
| 8 | Gradio UI | Modern themed interface |
| 9 | Launch | Start the app |

In [ ]:
# ============================================================
# Fix pip version so fairseq's old metadata parses correctly
# ============================================================
!pip install --quiet "pip<24.1"

import sys
print(f"Python: {sys.version.split()[0]}")
!pip --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 52.6 MB/s eta 0:00:00
Python: 3.12.13
pip 24.0 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)


## Cell 1 — Install all dependencies

In [ ]:
# ============================================================
# CELL 1A — System dependencies (run once)
# ============================================================
!sudo apt-get update -qq
!sudo apt-get install -y -qq gcc g++ make cmake ninja-build \
    libomp-dev libgsl-dev gsl-bin libgsl-dbg \
    python3-pybind11 pybind11-dev unicode libicu-dev 2>&1 | tail -3

print("✓ System dependencies installed")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✓ System dependencies installed


**Restart runtime now** (Runtime → Restart session), then run every cell below.

In [ ]:
# ============================================================
# CELL 1B — Download + install MorphoKIN (KINLP)
# ============================================================
import os, subprocess
from pathlib import Path

KINLP_DIR = Path("/opt/KINLP")

if not KINLP_DIR.exists():
    print("Downloading MorphoKIN (26 GB — takes 10-20 min)...")
    # Method 1: Direct gdown (usually works on Colab)
    !pip install -q gdown
    !gdown "1Kt9YXhLw_UVMCefcRGworyHUdh-tyQRj" -O /tmp/KINLP.tar.gz

    # Extract
    !cd /tmp && tar xzf KINLP.tar.gz
    # Find the extracted directory
    import glob
    kinlp_dirs = glob.glob("/tmp/KINLP*")
    kinlp_extracted = [d for d in kinlp_dirs if os.path.isdir(d) and d != "/tmp/KINLP.tar.gz"]
    if kinlp_extracted:
        !sudo ln -sf {kinlp_extracted[0]} /opt/KINLP
    else:
        print("❌ Could not find extracted KINLP directory")
        print(f"  Contents of /tmp: {os.listdir('/tmp')}")

    !rm -f /tmp/KINLP.tar.gz
    print("✓ MorphoKIN extracted")
else:
    print("✓ MorphoKIN already installed")

# Set environment variables
UBUNTU_VERSION = subprocess.check_output(["lsb_release", "-r", "--short"]).decode().strip()
os.environ["KINLP_HOME"] = "/opt/KINLP"
os.environ["PATH"] = f"{os.environ['PATH']}:/opt/KINLP:/opt/KINLP/bin/{UBUNTU_VERSION}"
os.environ["LD_LIBRARY_PATH"] = f"{os.environ.get('LD_LIBRARY_PATH', '')}:/opt/KINLP/lib/{UBUNTU_VERSION}"

print(f"  KINLP_HOME:       {os.environ['KINLP_HOME']}")
print(f"  Ubuntu version:   {UBUNTU_VERSION}")
print(f"  KINLP exists:     {KINLP_DIR.exists()}")
print(f"  KINLP contents:   {os.listdir('/opt/KINLP') if KINLP_DIR.exists() else 'N/A'}")

✓ MorphoKIN already installed
  KINLP_HOME:       /opt/KINLP
  Ubuntu version:   22.04
  KINLP exists:     True
  KINLP contents:   ['morphokin.sh', 'models', 'data', 'lib', 'bin', 'include', 'run', 'sample_docs', 'LICENSE.pdf']


In [ ]:
# ============================================================
# CELL 1C — Upload your license file
# ============================================================
from google.colab import files

# Upload your LICENSE_FILE.dat
LICENSE_PATH = "/content/morphokin_license.dat"

if not os.path.exists(LICENSE_PATH):
    print("Upload your MorphoKIN license file (.dat):")
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(LICENSE_PATH, "wb") as f:
            f.write(data)
        print(f"✓ License saved to {LICENSE_PATH}")
        break
else:
    print(f"✓ License already at {LICENSE_PATH}")

# Verify license
!bash /opt/KINLP/morphokin.sh license {LICENSE_PATH}

✓ License already at /content/morphokin_license.dat
MorphoKIN v1.0
Run: <kinya-morpho-app> <command> </path/to/LICENSE_FILE.dat>

=== COMMANDS OPTIONS ====
<kinya-morpho-app> rms </path/to/LICENSE_FILE.dat>	Run morpho-analysis-synthesis server on a unix socket
<kinya-morpho-app> snt </path/to/LICENSE_FILE.dat>	Interactive morphological parsing of text
<kinya-morpho-app> license </path/to/LICENSE_FILE.dat>	Check license validity

Successful license validation!
License key: I8DVFW
Starts on: Mon 2026-06-29 21:41:11 UTC
Expires on: Tue 2027-06-29 21:41:11 UTC



In [ ]:
# ============================================================
# CELL 1D — Start MorphoKIN server (background daemon)
# ============================================================
import subprocess, time

# Check if already running
result = subprocess.run(["pgrep", "-f", "morphokin"], capture_output=True)
if result.returncode == 0:
    print("✓ MorphoKIN server already running")
else:
    print("Starting MorphoKIN server (takes 30-60 seconds to initialize)...")
    proc = subprocess.Popen(
        ["bash", "/opt/KINLP/morphokin.sh", "rms", LICENSE_PATH],
        stdout=open("/content/morphokin_server.log", "w"),
        stderr=subprocess.STDOUT,
    )
    # Wait for it to initialize
    for i in range(120):  # max 2 minutes
        time.sleep(1)
        # Check if socket file exists (sign that server is ready)
        if os.path.exists("/tmp/morphokin.sock") or os.path.exists("/tmp/kinlp_morpho.sock"):
            print(f"✓ MorphoKIN server started (took {i+1}s)")
            break
        if proc.poll() is not None:
            print(f"❌ MorphoKIN server crashed. Check /content/morphokin_server.log")
            !cat /content/morphokin_server.log | tail -20
            break
    else:
        print("⚠️ MorphoKIN server startup timed out (120s). Checking logs...")
        !cat /content/morphokin_server.log | tail -20

Starting MorphoKIN server (takes 30-60 seconds to initialize)...
⚠️ MorphoKIN server startup timed out (120s). Checking logs...

2026-07-02 23:42:59	Parsing match/req rules...
total_pref_require_global_ids: 428
TOTAL_POS_PREFERENCE_RULES: 50
TOTAL_FEATURED_VERB_MORPHO_RULES: 174
Initiated all POS classes: 148

2026-07-02 23:42:59	Reading known lexicon...
	Special words: 401
	Verbs lemma: 8158
	Common verbs lemma: 1979
	Lemmatized verbs: 4518
	Nouns lemma: 17044
	Common nouns lemma: 4434

2026-07-02 23:42:59	Reading collected stem features...
	Collected verb stem features: 3429

2026-07-02 23:43:00	Initializing word vectors...


In [ ]:
# ============================================================
# Install DeepKIN with correct dependency versions
# ============================================================
import subprocess, sys, os
from pathlib import Path

DEEPKIN_PATH = Path("/content/DeepKIN")

# Clone if not present
if not DEEPKIN_PATH.exists():
    print("Cloning DeepKIN...")
    !git clone --quiet https://github.com/anzeyimana/DeepKIN.git {DEEPKIN_PATH}
else:
    print(f"✓ Repo already at {DEEPKIN_PATH}")

# Install DeepKIN's implicit dependencies WITH pinned compatible versions
print("\nInstalling pinned deps (omegaconf + hydra-core versions compatible with fairseq)...")
!pip install --quiet "omegaconf==2.0.6" "hydra-core==1.0.7" 2>&1 | tail -3

# Now install fairseq — it should work with pip<24.1
print("\nInstalling fairseq...")
!pip install --quiet fairseq==0.12.2 2>&1 | tail -5

# Other DeepKIN deps
print("\nInstalling other DeepKIN deps...")
!pip install --quiet Cython distro progressbar2 seqeval youtokentome tensorboardX sacremoses fastBPE mutagen torchmetrics pandas 2>&1 | tail -3

# Install DeepKIN itself
print("\nInstalling DeepKIN in editable mode...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "."],
    cwd=str(DEEPKIN_PATH),
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(f"⚠️ Install returned {result.returncode}")
    print(result.stderr[-1500:])
else:
    print("✓ DeepKIN pip install completed")

# Test imports
print("\n" + "="*60)
try:
    from deepkin.data.kinya_norm import text_to_sequence
    from deepkin.models.flex_tts import FlexKinyaTTS
    from deepkin.modules.tts_commons import intersperse
    print("✅ DeepKIN imports work — you can now run Cell 6 (TTS loading)")
except ImportError as e:
    print(f"❌ Import still failing: {e}")

Cloning DeepKIN...

Installing pinned deps (omegaconf + hydra-core versions compatible with fairseq)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 9.1 MB/s eta 0:00:00
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

Installing fairseq...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.0/344.0 kB 9.5 MB/s eta 0:00:00
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version

In [ ]:
# ============================================================
# CELL 1E — Install DeepKIN + dependencies
# ============================================================

# Core Python dependencies
!pip install -q Cython distro progressbar2 seqeval youtokentome \
    tensorboardX sacremoses fastBPE packaging mutagen torchmetrics pandas

# fairseq (needed by DeepKIN)
!pip install -q fairseq 2>/dev/null || (git clone https://github.com/pytorch/fairseq /tmp/fairseq && pip install -e /tmp/fairseq)

# NVIDIA apex — the tricky one
# Try the pip version first (faster, sometimes works)
!pip install -q apex 2>/dev/null || echo "pip apex failed, trying source build..."

# If pip install didn't work, build from source
import importlib
try:
    importlib.import_module("apex")
    print("✓ apex available via pip")
except ImportError:
    print("Building apex from source (10-20 min)...")
    !git clone https://github.com/NVIDIA/apex /tmp/apex 2>/dev/null
    !cd /tmp/apex && pip install -v --disable-pip-version-check --no-cache-dir --no-build-isolation \
        --config-settings "--build-option=--cpp_ext" \
        --config-settings "--build-option=--cuda_ext" ./ 2>&1 | tail -5

# DeepKIN itself
!git clone https://github.com/anzeyimana/DeepKIN.git /content/DeepKIN 2>/dev/null
!cd /content/DeepKIN && pip install -e ./ 2>&1 | tail -5

# Verify imports
try:
    from deepkin.data.kinya_norm import text_to_sequence
    from deepkin.modules.tts_commons import intersperse
    print("✓ DeepKIN imports successful")
except ImportError as e:
    print(f"❌ DeepKIN import failed: {e}")

DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
Building apex from source (10-20 min)...
    Uninstalling apex-0.9.10.dev0:
      Removing file or directory /usr/local/lib/python3.12/dist-packages/apex-0.9.10.dev0.dist-info/
      Removing file or directory /usr/local/lib/python3.12/dist-packages/apex/
      Successfully uninstalled apex-0.9.10.dev0
    Found existing installation: deepkin 1.0.0
    Uninstalling deepkin-1.0.0:
      Successfully uninstalled deepkin-1.0.0
  Running setup.py develop for deepkin
❌ DeepKIN import failed: No module named 'deepkin'


In [ ]:
pip install "git+https://github.com/c4ir-rw/ac-ai-models.git#subdirectory=DeepKIN-AgAI"

  Cloning https://github.com/c4ir-rw/ac-ai-models.git to /tmp/pip-req-build-twhv7qa9
  Running command git clone --filter=blob:none --quiet https://github.com/c4ir-rw/ac-ai-models.git /tmp/pip-req-build-twhv7qa9
  Resolved https://github.com/c4ir-rw/ac-ai-models.git to commit c28db6bfb6b80666583717c87e8edaa2a8f384d5
  Preparing metadata (setup.py) ... done
  Created wheel for deepkin: filename=deepkin-1.0.0-py3-none-any.whl size=184486 sha256=c7e7236d0a042a35af0047504bb5a0b83f4725c5c4c91f2160b321a9561041ad
  Stored in directory: /tmp/pip-ephem-wheel-cache-_dh524wm/wheels/aa/eb/b5/a0546a087db935bdab8ed4466935cd56cb5f1fe9f2bd16dbae
Successfully built deepkin
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://

In [ ]:
pip install typed-argument-parser==1.11.0

DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [ ]:
# ============================================================
# CELL 1F — Download + test kinya-flex-tts
# ============================================================
import torch
from huggingface_hub import hf_hub_download

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Downloading kinya-flex-tts checkpoint...")
ckpt = hf_hub_download(
    repo_id="C4IR-RW/kinya-flex-tts",
    filename="kinya_flex_tts_base_trained.pt",
)
print(f"✓ Checkpoint at: {ckpt}")

# Load the model
from deepkin.models.flex_tts import FlexKinyaTTS
from deepkin.data.kinya_norm import text_to_sequence
from deepkin.modules.tts_commons import intersperse

tts_model = FlexKinyaTTS.from_pretrained(DEVICE, ckpt)
tts_model.eval()

# Test inference
test_text = "Muraho, murakomeye?"
sequence = intersperse(text_to_sequence(test_text, norm=True), 0)

with torch.no_grad():
    audio = tts_model(sequence, 0)  # speaker 0 = Female 1

print(f"✓ kinya-flex-tts working!")
print(f"  Test text:    '{test_text}'")
print(f"  Audio shape:  {audio.shape}")
print(f"  Audio dtype:  {audio.dtype}")
print(f"  Duration:     {audio.shape[-1]/24000:.2f} seconds")

# Play it
import torchaudio
from IPython.display import Audio, display
audio_np = audio.cpu().squeeze().numpy()
display(Audio(audio_np, rate=24000))
print("\n🎉 If you heard 'Muraho, murakomeye?' above, kinya-flex-tts is fully working!")

kinya_flex_tts_base_trained.pt:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

✓ Checkpoint at: /root/.cache/huggingface/hub/models--C4IR-RW--kinya-flex-tts/snapshots/b4442362fba265bfa4da47420b405d78e5239a99/kinya_flex_tts_base_trained.pt
2026-07-03 00:00:54 Loading FlexTTS from /root/.cache/huggingface/hub/models--C4IR-RW--kinya-flex-tts/snapshots/b4442362fba265bfa4da47420b405d78e5239a99/kinya_flex_tts_base_trained.pt
2026-07-03 00:01:02 FlexTTS train steps: 2,000K
2026-07-03 00:01:02 Loading FlexTTS from /root/.cache/huggingface/hub/models--C4IR-RW--kinya-flex-tts/snapshots/b4442362fba265bfa4da47420b405d78e5239a99/kinya_flex_tts_base_trained.pt done!
✓ kinya-flex-tts working!
  Test text:    'Muraho, murakomeye?'
  Audio shape:  torch.Size([1, 35840])
  Audio dtype:  torch.float32
  Duration:     1.49 seconds



🎉 If you heard 'Muraho, murakomeye?' above, kinya-flex-tts is fully working!


## Cell 2 — Configuration

In [ ]:
import os
from pathlib import Path

# ── Project paths ─────────────────────────────────────────────
BASE_DIR   = Path("/content/ubuzima_ai")
CACHE_DIR  = BASE_DIR / "cache"
OUTPUT_DIR = BASE_DIR / "outputs"
TTS_DIR    = BASE_DIR / "models" / "kinya-flex-tts"

for d in [CACHE_DIR, OUTPUT_DIR, TTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Model IDs ─────────────────────────────────────────────────
ASR_MODEL_ID = "akera/whisper-large-v3-kin-200h-v2"
TTS_HF_REPO  = "C4IR-RW/kinya-flex-tts"
TTS_FALLBACK = "facebook/mms-tts-kin"
CLAUDE_MODEL = "claude-sonnet-4-6"

# ── ASR ───────────────────────────────────────────────────────
LANGUAGE       = "swahili"          # closest Bantu language Whisper knows
SAMPLE_RATE    = 16000
MAX_AUDIO_SECS = 30

# ── TTS ───────────────────────────────────────────────────────
TTS_SAMPLE_RATE  = 24000            # kinya-flex-tts native rate
TTS_FALLBACK_SR  = 16000            # MMS-TTS rate

SPEAKERS = {
    "Umugore 1 (Female 1)": 0,
    "Umugore 2 (Female 2)": 1,
    "Umugabo (Male)":       2,
}
DEFAULT_SPEAKER = 0

# ── LLM ───────────────────────────────────────────────────────
MAX_TOKENS  = 500
TEMPERATURE = 0.3

# ── Device ────────────────────────────────────────────────────
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float16 if DEVICE.type == "cuda" else torch.float32

print(f"✓ Config loaded")
print(f"  Device:    {DEVICE}")
print(f"  ASR:       {ASR_MODEL_ID}")
print(f"  LLM:       {CLAUDE_MODEL}")
print(f"  TTS:       {TTS_HF_REPO} (fallback: {TTS_FALLBACK})")
print(f"  Output:    {OUTPUT_DIR}")


✓ Config loaded
  Device:    cuda
  ASR:       akera/whisper-large-v3-kin-200h-v2
  LLM:       claude-sonnet-4-6
  TTS:       C4IR-RW/kinya-flex-tts (fallback: facebook/mms-tts-kin)
  Output:    /content/ubuzima_ai/outputs


## Cell 3 — Authentication

In [ ]:
!pip install -q google-generativeai

DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"  # Replace with your actual API key

## Cell 4 — ASR module (`SpeechRecognizer`)

Wraps `akera/whisper-large-v3-kin-200h-v2` in a clean class with one method: `transcribe(audio) → str`.

In [ ]:
import numpy as np
import soundfile as sf
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration, GenerationConfig


class SpeechRecognizer:
    """Kinyarwanda ASR using akera Whisper-large-v3."""

    def __init__(self):
        print(f"Loading ASR: {ASR_MODEL_ID}...")

        self.processor = WhisperProcessor.from_pretrained(ASR_MODEL_ID)
        self.processor.tokenizer.set_prefix_tokens(
            language=LANGUAGE, task="transcribe"
        )

        self.model = WhisperForConditionalGeneration.from_pretrained(
            ASR_MODEL_ID, torch_dtype=DTYPE,
        ).to(DEVICE).eval()

        # Fix outdated generation config from akera checkpoint
        self.model.generation_config = GenerationConfig.from_pretrained(
            "openai/whisper-large-v3"
        )

        self.model.config.forced_decoder_ids = None
        self.model.config.suppress_tokens = []

        # Precompute forced decoder IDs for the language token
        self.forced_decoder_ids = self.processor.get_decoder_prompt_ids(
            language=LANGUAGE, task="transcribe"
        )

        print("✓ ASR loaded")

    def transcribe(self, audio_tuple) -> str:
        """
        Accepts Gradio audio tuple (sample_rate, numpy_array).
        Returns transcribed Kinyarwanda text.
        """
        if audio_tuple is None:
            return ""

        sr, arr = audio_tuple
        arr = np.asarray(arr, dtype=np.float32)

        # Mono
        if arr.ndim > 1:
            arr = arr.mean(axis=1)

        # Normalize int16 → float
        if np.max(np.abs(arr)) > 1.0:
            arr = arr / 32768.0

        # Resample to 16 kHz
        if sr != SAMPLE_RATE:
            arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLE_RATE)

        # Featurize + generate
        inputs = self.processor.feature_extractor(
            arr, sampling_rate=SAMPLE_RATE, return_tensors="pt"
        ).to(DEVICE, dtype=DTYPE)

        with torch.no_grad():
            ids = self.model.generate(
                inputs.input_features,
                max_length=225,
                forced_decoder_ids=self.forced_decoder_ids,
            )

        text = self.processor.batch_decode(ids, skip_special_tokens=True)[0]
        return text.strip()


# Instantiate
asr = SpeechRecognizer()

Loading ASR: akera/whisper-large-v3-kin-200h-v2...


preprocessor_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

✓ ASR loaded


## Cell 5 — LLM module (`ClaudeAssistant`)

Uses Anthropic Claude API as the "Brain" — generates health answers in Kinyarwanda.

In [ ]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 9.5 MB/s eta 0:00:00
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [ ]:
import google.generativeai as genai
import os

# Configure once
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])


SYSTEM_PROMPT = """
Witwa UBUZIMA AI. Uri umufasha mu by'ubuzima utanga inama z'ibanze mu Kinyarwanda.

AMABWIRIZA:
1. Subiza mu Kinyarwanda gusa, mu nteruro 2-3 zoroshye. Ntukarenze amagambo 60.
"""

class ClaudeAssistant:
    """Health Q&A using Google Gemini (Kinyarwanda-capable, free tier)."""

    def __init__(self):
        self.model = genai.GenerativeModel(
            model_name="gemini-2.5-flash",
            system_instruction=SYSTEM_PROMPT.strip(),
            generation_config={
                "temperature": 0.3,
                "max_output_tokens": 1024,
            },
        )
        print(f"✓ Gemini client ready (model=gemini-2.5-flash)")

    def ask(self, question: str) -> str:
        """Kinyarwanda question → Kinyarwanda answer."""
        response = self.model.generate_content(question)
        return response.text.strip()


# Instantiate
llm = ClaudeAssistant()

✓ Gemini client ready (model=gemini-2.5-flash)


## Cell 6 — TTS module (`TextToSpeech`)

Tries to load **C4IR-RW/kinya-flex-tts** (3-speaker, 24 kHz, IVR-quality) first. If DeepKIN or MorphoKIN is unavailable, falls back to **MMS-TTS** (single speaker, 16 kHz). Both produce clean Kinyarwanda speech.

In [ ]:
import torchaudio
from datetime import datetime


class KinyaFlexTTS:
    """C4IR-RW kinya-flex-tts — 3-speaker Kinyarwanda TTS."""

    def __init__(self):
        from huggingface_hub import hf_hub_download
        from deepkin.data.kinya_norm import text_to_sequence
        from deepkin.models.flex_tts import FlexKinyaTTS
        from deepkin.modules.tts_commons import intersperse

        print(f"Loading kinya-flex-tts from {TTS_HF_REPO}...")

        # Download checkpoint
        ckpt_path = hf_hub_download(
            repo_id=TTS_HF_REPO,
            filename="kinya_flex_tts_base_trained.pt",
            cache_dir=str(CACHE_DIR),
        )

        self.model = FlexKinyaTTS.from_pretrained(DEVICE, ckpt_path)
        self.model.eval()
        self.text_to_sequence = text_to_sequence
        self.intersperse = intersperse
        self.sample_rate = TTS_SAMPLE_RATE
        self.supports_speakers = True

        print("✓ kinya-flex-tts loaded (3-speaker, 24 kHz)")

    def synthesize(self, text: str, speaker: int = DEFAULT_SPEAKER):
        """Text → (sample_rate, numpy waveform)."""
        sequence = self.intersperse(
            self.text_to_sequence(text, norm=True), 0,
        )
        with torch.no_grad():
            waveform = self.model(sequence, speaker)

        if hasattr(waveform, "cpu"):
            waveform = waveform.cpu()
        audio = waveform.squeeze().numpy().astype(np.float32)
        return (self.sample_rate, audio)

    def save(self, text: str, speaker: int = DEFAULT_SPEAKER) -> Path:
        """Text → saved WAV file path."""
        sr, audio = self.synthesize(text, speaker)
        filename = datetime.now().strftime("%Y%m%d_%H%M%S.wav")
        out = OUTPUT_DIR / filename
        sf.write(str(out), audio, sr)
        return out


class MMSFallbackTTS:
    """Meta MMS-TTS Kinyarwanda — single-speaker fallback."""

    def __init__(self):
        from transformers import VitsModel, AutoTokenizer

        print(f"Loading MMS-TTS fallback: {TTS_FALLBACK}...")

        self.tokenizer = AutoTokenizer.from_pretrained(TTS_FALLBACK)
        self.model = VitsModel.from_pretrained(TTS_FALLBACK).to(DEVICE).eval()
        self.sample_rate = TTS_FALLBACK_SR
        self.supports_speakers = False

        print("✓ MMS-TTS loaded (single speaker, 16 kHz)")

    def synthesize(self, text: str, speaker: int = 0):
        """Text → (sample_rate, numpy waveform). Speaker arg ignored."""
        inputs = self.tokenizer(text, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            waveform = self.model(**inputs).waveform[0]
        audio = waveform.cpu().numpy().astype(np.float32)
        return (self.sample_rate, audio)

    def save(self, text: str, speaker: int = 0) -> Path:
        sr, audio = self.synthesize(text, speaker)
        filename = datetime.now().strftime("%Y%m%d_%H%M%S.wav")
        out = OUTPUT_DIR / filename
        sf.write(str(out), audio, sr)
        return out


# ── Load TTS with fallback ────────────────────────────────────
tts = None
tts_backend = "none"

try:
    tts = KinyaFlexTTS()
    tts_backend = "kinya-flex-tts"
except Exception as e:
    print(f"⚠️ kinya-flex-tts unavailable: {type(e).__name__}: {str(e)[:200]}")
    print("   Falling back to MMS-TTS...")
    try:
        tts = MMSFallbackTTS()
        tts_backend = "mms-tts"
    except Exception as e2:
        print(f"❌ MMS-TTS also failed: {e2}")
        tts_backend = "none"

print(f"\n✓ TTS backend: {tts_backend}")
if tts and tts.supports_speakers:
    print(f"  Available voices: {list(SPEAKERS.keys())}")


Loading kinya-flex-tts from C4IR-RW/kinya-flex-tts...


kinya_flex_tts_base_trained.pt:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

2026-07-03 00:08:53 Loading FlexTTS from /content/ubuzima_ai/cache/models--C4IR-RW--kinya-flex-tts/snapshots/b4442362fba265bfa4da47420b405d78e5239a99/kinya_flex_tts_base_trained.pt
2026-07-03 00:08:55 FlexTTS train steps: 2,000K
2026-07-03 00:08:55 Loading FlexTTS from /content/ubuzima_ai/cache/models--C4IR-RW--kinya-flex-tts/snapshots/b4442362fba265bfa4da47420b405d78e5239a99/kinya_flex_tts_base_trained.pt done!
✓ kinya-flex-tts loaded (3-speaker, 24 kHz)

✓ TTS backend: kinya-flex-tts
  Available voices: ['Umugore 1 (Female 1)', 'Umugore 2 (Female 2)', 'Umugabo (Male)']


## Cell 7 — Pipeline orchestrator

`UbuzimaPipeline` ties ASR → LLM → TTS together with per-stage latency tracking.

In [ ]:
import time


class Timer:
    """Context manager for timing pipeline stages."""
    def __init__(self):
        self.elapsed = 0.0
    def __enter__(self):
        self._t0 = time.perf_counter()
        return self
    def __exit__(self, *args):
        self.elapsed = round(time.perf_counter() - self._t0, 3)


class UbuzimaPipeline:
    """Full voice pipeline: audio → transcript → answer → speech."""

    def __init__(self, asr, llm, tts):
        self.asr = asr
        self.llm = llm
        self.tts = tts

    def run_voice(self, audio_tuple, speaker=DEFAULT_SPEAKER):
        """Audio input → (transcript, answer, audio_out, latency_str)."""
        if audio_tuple is None:
            return "⚠️ Nta majwi yumvikanye", "", None, ""

        latency = {}

        # ASR
        with Timer() as t:
            transcript = self.asr.transcribe(audio_tuple)
        latency["asr"] = t.elapsed

        if not transcript.strip():
            return "⚠️ Sinumva neza — ongera uvuge", "", None, ""

        # LLM
        with Timer() as t:
            answer = self.llm.ask(transcript)
        latency["llm"] = t.elapsed

        # TTS
        audio_out = None
        if self.tts:
            with Timer() as t:
                audio_out = self.tts.synthesize(answer, speaker)
            latency["tts"] = t.elapsed

        latency["total"] = round(sum(latency.values()), 3)
        lat_str = " · ".join(f"{k.upper()} {v*1000:.0f}ms" for k, v in latency.items())

        return transcript, answer, audio_out, f"⏱ {lat_str}"

    def run_text(self, question, speaker=DEFAULT_SPEAKER):
        """Text input → (question, answer, audio_out, latency_str)."""
        if not question or not question.strip():
            return "", "", None, ""

        latency = {}

        # LLM
        with Timer() as t:
            answer = self.llm.ask(question)
        latency["llm"] = t.elapsed

        # TTS
        audio_out = None
        if self.tts:
            with Timer() as t:
                audio_out = self.tts.synthesize(answer, speaker)
            latency["tts"] = t.elapsed

        latency["total"] = round(sum(latency.values()), 3)
        lat_str = " · ".join(f"{k.upper()} {v*1000:.0f}ms" for k, v in latency.items())

        return question, answer, audio_out, f"⏱ {lat_str}"


# Instantiate
pipeline = UbuzimaPipeline(asr=asr, llm=llm, tts=tts)
print("✓ Pipeline ready")


✓ Pipeline ready


In [ ]:
pipeline = UbuzimaPipeline(asr=asr, llm=llm, tts=tts)
print("✓ Pipeline reloaded with Gemini LLM")

✓ Pipeline reloaded with Gemini LLM


In [ ]:
try:
    resp = llm.ask("Muraho, ni iki gikora malariya?")
    print(f"✓ Gemini responded ({len(resp)} chars):")
    print(f"  {resp}")
except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")

## Cell 8 — Gradio UI (modern themed interface)

Two tabs: **Voice** (record → transcribe → answer → speak) and **Text** (type/click example → answer → speak).

Features: speaker selector dropdown, latency metrics, 6 example health questions, auto-playing audio, custom teal/amber theme.

In [ ]:
import gradio as gr

# ── Custom CSS ────────────────────────────────────────────────
CSS = """
/* Container */
.gradio-container {
    max-width: 960px !important;
    margin: 0 auto !important;
    font-family: 'Segoe UI', system-ui, -apple-system, sans-serif !important;
}

/* Hero banner */
.hero-banner {
    background: linear-gradient(135deg, #00695C 0%, #004D40 60%, #00897B 100%);
    color: white;
    padding: 32px 24px;
    border-radius: 20px;
    text-align: center;
    margin-bottom: 20px;
    box-shadow: 0 8px 32px rgba(0,77,64,0.3);
}
.hero-banner h1 {
    font-size: 2.6em;
    font-family: Cambria, Georgia, serif;
    margin: 0;
    letter-spacing: 1px;
}
.hero-banner .subtitle {
    color: #B2DFDB;
    font-style: italic;
    margin: 6px 0 14px;
    font-size: 1.05em;
}
.hero-banner .status-row {
    display: flex;
    justify-content: center;
    gap: 8px;
    flex-wrap: wrap;
}
.pill {
    display: inline-block;
    background: rgba(255,255,255,0.15);
    backdrop-filter: blur(4px);
    padding: 5px 14px;
    border-radius: 20px;
    font-size: 0.78em;
    letter-spacing: 0.3px;
}

/* Section cards */
.section-card {
    background: #FAFFFE;
    border: 1px solid #E0F2F1;
    border-radius: 14px;
    padding: 20px;
    margin: 8px 0;
}

/* Example buttons */
.example-btn {
    background: #F1F8F6 !important;
    border: 1.5px solid #C8E6C9 !important;
    border-radius: 12px !important;
    padding: 12px !important;
    text-align: left !important;
    transition: all 0.2s ease !important;
    font-size: 0.92em !important;
}
.example-btn:hover {
    border-color: #00695C !important;
    background: #E0F2F1 !important;
    transform: translateY(-1px) !important;
    box-shadow: 0 4px 12px rgba(0,105,92,0.12) !important;
}

/* Results styling */
.result-question {
    border-left: 4px solid #FFA726 !important;
    background: #FFF8E1 !important;
    border-radius: 0 10px 10px 0 !important;
}
.result-answer {
    border-left: 4px solid #00695C !important;
    background: #E0F2F1 !important;
    border-radius: 0 10px 10px 0 !important;
}

/* Footer */
.app-footer {
    text-align: center;
    color: #78909C;
    font-size: 0.78em;
    margin-top: 24px;
    padding: 16px;
    border-top: 1px solid #ECEFF1;
}
.app-footer em { color: #B0BEC5; }

/* Record button pulse */
.record-area .gradio-button.primary {
    background: #00695C !important;
    border-radius: 50% !important;
    width: 80px !important;
    height: 80px !important;
}

/* Tab styling */
.tab-nav button { font-weight: 600 !important; }
.tab-nav button.selected {
    border-color: #00695C !important;
    color: #00695C !important;
}
"""


# ── Examples ──────────────────────────────────────────────────
EXAMPLES = [
    ("🦟", "Ni iki gikora malariya?",               "Malaria"),
    ("🌡️", "Umwana wanjye afite umuriro, ni iki nakora?", "Umuriro w'umwana"),
    ("🤧", "Uburyo bwo kwirinda indwara z'ubuhumekero?",  "Ubuhumekero"),
    ("💧", "Ni ryari nagomba kunywa amazi menshi?",        "Amazi"),
    ("🤕", "Mfite umutwe ubabaza, nakora iki?",           "Umutwe"),
    ("🤰", "Umugore utwite agomba kurya iki?",            "Gusama"),
]


# ── Speaker dropdown options ─────────────────────────────────
speaker_choices = list(SPEAKERS.keys()) if tts and tts.supports_speakers else ["Default"]


# ── Wrappers for Gradio ──────────────────────────────────────
def voice_handler(audio, speaker_name, progress=gr.Progress()):
    speaker_id = SPEAKERS.get(speaker_name, DEFAULT_SPEAKER)
    progress(0.15, desc="🎧 Ndumva ikibazo cyawe...")
    transcript, answer, audio_out, latency = pipeline.run_voice(audio, speaker_id)
    progress(1.0, desc="✓ Byarangiye")
    return transcript, answer, audio_out, latency


def text_handler(question, speaker_name, progress=gr.Progress()):
    speaker_id = SPEAKERS.get(speaker_name, DEFAULT_SPEAKER)
    progress(0.3, desc="💭 Ndategura igisubizo...")
    _, answer, audio_out, latency = pipeline.run_text(question, speaker_id)
    progress(1.0, desc="✓ Byarangiye")
    return answer, audio_out, latency


def example_handler(emoji, question, label, speaker_name):
    return text_handler(question, speaker_name)


# ── Build app ─────────────────────────────────────────────────
with gr.Blocks(css=CSS, title="UBUZIMA AI", theme=gr.themes.Soft()) as app:

    # Hero banner
    tts_label = "kinya-flex-tts (3 voices)" if tts_backend == "kinya-flex-tts" else "MMS-TTS"
    gr.HTML(f"""
        <div class="hero-banner">
            <h1>🩺 UBUZIMA AI</h1>
            <p class="subtitle">Umufasha mu by'ubuzima uvuga Ikinyarwanda</p>
            <div class="status-row">
                <span class="pill">● ASR: akera Whisper</span>
                <span class="pill">● LLM: Claude (Anthropic)</span>
                <span class="pill">● TTS: {tts_label}</span>
                <span class="pill">● GPU: {'✓ ' + str(DEVICE) if DEVICE.type == 'cuda' else '✗ CPU'}</span>
            </div>
        </div>
    """)

    # Speaker selector (shared across tabs)
    with gr.Row():
        speaker_dropdown = gr.Dropdown(
            choices=speaker_choices,
            value=speaker_choices[0],
            label="🗣️ Hitamo ijwi / Choose voice",
            interactive=tts is not None and tts.supports_speakers,
            scale=2,
        )
        gr.Markdown("", scale=3)  # spacer

    # ── TAB 1: VOICE ──────────────────────────────────────
    with gr.Tab("🎙️ Ijwi (Voice)", id="voice"):
        gr.HTML('<div class="section-card">')
        gr.Markdown("### Kanda ku buto ya microphone, uvuge ikibazo cyawe mu Kinyarwanda")

        with gr.Row():
            with gr.Column(scale=1):
                voice_input = gr.Audio(
                    sources=["microphone"],
                    type="numpy",
                    label="🎤 Vuga hano",
                )
                voice_btn = gr.Button(
                    "✨ Kora igisubizo",
                    variant="primary",
                    size="lg",
                )

            with gr.Column(scale=2):
                voice_transcript = gr.Textbox(
                    label="📝 Ikibazo cyawe (transcript)",
                    lines=2,
                    elem_classes=["result-question"],
                )
                voice_answer = gr.Textbox(
                    label="💬 Igisubizo",
                    lines=5,
                    elem_classes=["result-answer"],
                )
                voice_audio = gr.Audio(
                    label="🔊 Ijwi ry'igisubizo",
                    autoplay=True,
                )
                voice_latency = gr.Markdown("")

        gr.HTML('</div>')

        voice_btn.click(
            voice_handler,
            inputs=[voice_input, speaker_dropdown],
            outputs=[voice_transcript, voice_answer, voice_audio, voice_latency],
        )

    # ── TAB 2: TEXT ────────────────────────────────────────
    with gr.Tab("📝 Inyandiko (Text)", id="text"):
        gr.HTML('<div class="section-card">')
        gr.Markdown("### Andika ikibazo cyawe cyangwa uhitemo kimwe mu bibazo bikurikira")

        with gr.Row():
            with gr.Column(scale=1):
                text_input = gr.Textbox(
                    label="✍️ Ikibazo cyawe",
                    placeholder="Urugero: Ni iki gikora malariya?",
                    lines=3,
                )
                text_btn = gr.Button(
                    "✨ Kora igisubizo",
                    variant="primary",
                    size="lg",
                )

                gr.Markdown("#### 🏥 Ibibazo bishoboka")
                with gr.Column():
                    ex_btns = []
                    for emoji, question, label in EXAMPLES:
                        btn = gr.Button(
                            f"{emoji}  {question}",
                            elem_classes=["example-btn"],
                        )
                        ex_btns.append((btn, question))

            with gr.Column(scale=2):
                text_question_display = gr.Textbox(
                    label="📝 Ikibazo",
                    lines=2,
                    elem_classes=["result-question"],
                    value="",
                )
                text_answer = gr.Textbox(
                    label="💬 Igisubizo",
                    lines=6,
                    elem_classes=["result-answer"],
                )
                text_audio = gr.Audio(
                    label="🔊 Ijwi ry'igisubizo",
                    autoplay=True,
                )
                text_latency = gr.Markdown("")

        gr.HTML('</div>')

        # Wire text submit button
        def text_submit(question, speaker):
            answer, audio, latency = text_handler(question, speaker)
            return question, answer, audio, latency

        text_btn.click(
            text_submit,
            inputs=[text_input, speaker_dropdown],
            outputs=[text_question_display, text_answer, text_audio, text_latency],
        )

        # Wire example buttons
        for btn, question in ex_btns:
            def make_handler(q):
                def handler(speaker):
                    answer, audio, latency = text_handler(q, speaker)
                    return q, q, answer, audio, latency
                return handler
            handler_fn = make_handler(question)
            btn.click(
                handler_fn,
                inputs=[speaker_dropdown],
                outputs=[text_input, text_question_display, text_answer, text_audio, text_latency],
            )

    # ── TAB 3: ABOUT ──────────────────────────────────────
    with gr.Tab("ℹ️ About"):
        gr.Markdown(f"""
### URURIMI / UBUZIMA AI

**Capstone project** by Ganza Didier, BSc Software Engineering (Data Science & ML), African Leadership University, Kigali.

**Supervisor:** Emmanuel Adjei

---

#### Architecture

```
🎤 User speaks Kinyarwanda
    → akera Whisper ASR (speech → text)
    → Anthropic Claude (question → Kinyarwanda answer)
    → {'C4IR-RW kinya-flex-tts' if tts_backend == 'kinya-flex-tts' else 'Meta MMS-TTS'} (text → spoken Kinyarwanda)
🔊 User hears the answer
```

#### Tech Stack

| Layer | Technology |
|---|---|
| ASR | `{ASR_MODEL_ID}` |
| LLM | Anthropic Claude (`{CLAUDE_MODEL}`) |
| TTS | `{TTS_HF_REPO}` {'(active)' if tts_backend == 'kinya-flex-tts' else '(unavailable → MMS-TTS fallback)'} |
| UI | Gradio |
| Compute | Google Colab ({'GPU: ' + str(DEVICE) if DEVICE.type == 'cuda' else 'CPU'}) |

#### Credits

- ASR: [akera](https://huggingface.co/akera/whisper-large-v3-kin-200h-v2)
- TTS: [C4IR Rwanda](https://huggingface.co/C4IR-RW) + [KiNLP](https://kinlp.com/)
- LLM: [Anthropic](https://www.anthropic.com)

---

⚠️ *Iri gikoresho ntabwo risimbura muganga. Iyo ufite ikibazo gikomeye, jya kwa muganga.*
""")

    # Footer
    gr.HTML("""
        <div class="app-footer">
            <p>URURIMI / UBUZIMA AI · ALU Kigali · Ganza Didier · Supervisor: Emmanuel Adjei</p>
            <p><em>Iri gikoresho ntabwo risimbura muganga. Iyo ufite ikibazo gikomeye, jya kwa muganga.</em></p>
        </div>
    """)

print("✓ Gradio app built")


✓ Gradio app built


## Cell 9 — Launch the app

This starts the Gradio server. On Colab, it generates a public share URL automatically.

In [ ]:
# Launch with share=True for public URL
app.launch(
    share=True,
    debug=True,
    show_error=True,
    server_name="0.0.0.0",
    server_port=7860,
)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://125faf2f686e586296.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7860 <> https://125faf2f686e586296.gradio.live


## After launch

1. Copy the **public URL** from the output (looks like `https://xxxxx.gradio.live`)
2. Test all 6 example questions
3. Test voice recording
4. Record your demo video
5. When done, stop this cell to release the GPU